In [ ]:
%matplotlib inline

# Step-based Training Notebook

This notebook is for training models in a step-based fashion. This is useful for large datasets in which the amount of training samples makes epoch-based training impractical.

# User settings, and HDF5 File Loading

This next code block will handle creating the test DataLoader for inference. This requires a few steps.

1. This code block will filter and calculate normalization stats for the HDF5 file.
2. It will then create a custom dataset object containing both the HDF5 inputs and outputs.
3. It will then split the train, validation, and train datasets (The seed is set to 0 so this won't vary between runs).

In [ ]:
import os
import numpy as np
from math import exp
from datetime import datetime
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter


from data_utils import prepare_dataloaders
from U_FNO import UFNO3d
from AU_NET import UNet3DAttention

# ser Settings
BATCH_SIZE = 2
NORMALIZATION_METHOD = 'z-score'  # 'z-score' or 'min-max'
INPUT_HDF5 = 'small_input_arrays.hdf5'
OUTPUT_HDF5 = 'small_output_arrays.hdf5'
STATS_PATH = 'normalization_stats'
MAX_STEPS = 200000  # Total number of training steps
VALIDATION_FREQUENCY = 500  # Run validation every N steps
SAVE_FREQUENCY = 1000  # Save a model snapshot every N steps
CHECKPOINT_PATH = 'checkpoints_ufno'
PLOT_DIR = 'bad_samples' # where to put bad sample images for analysis
LOSS_THRESHOLD = 0.75
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

training_loader, validation_loader, test_loader = prepare_dataloaders(
    input_hdf5=INPUT_HDF5,
    output_hdf5=OUTPUT_HDF5,
    batch_size=BATCH_SIZE,
    norm_method=NORMALIZATION_METHOD,
    stats_path=STATS_PATH
)

print(f"DataLoaders are ready. Train size: {len(training_loader.dataset)}")

The Models
=========

Models are now created. Change `model_type` to `UFNOB` or `AUNET` depending on which model you want to train. The `modeN` hyperparameters pertain only to UFNO and can be optimized through hyperparameter tuning. All other hyperparameters pertain to both models.

In [ ]:
# model hyperparameters
model_type = 'UFNO'
mode1 = 16
mode2 = 12
mode3 = 4
width = 48
in_channels = 6
out_channels = 2

# create model
if model_type == 'AUNET':
    model = UNet3DAttention(
            in_channels=in_channels, 
            out_channels=out_channels,
            base_c=width
        )
elif model_type == 'UFNO':
    UNet = True
    if UNet: 
        model = UFNO3d(mode1, mode2, mode3, width, in_channels=in_channels, out_channels=out_channels, UNet = True)
    else:
        model = UFNO3d(mode1, mode2, mode3, width, in_channels=in_channels, out_channels=out_channels, UNet = False)

print(model)
model.to(device)
pytorch_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total params")
print(pytorch_total_params)

# Loss Function Definitions

This code block defines a number of loss functions that can be applied to either or both channels. See the end of the code block for examples on creating the combined loss functions. Some loss functions not used in Morphew et al. 2026 are included.

In [ ]:
# ==========================================================================================
# SECTION 1: INDIVIDUAL LOSS FUNCTION COMPONENTS
# These are the building blocks. Each operates on a given tensor.
# ==========================================================================================

class WeightedMSELoss(nn.Module):
    """
    Computes a weighted Mean Squared Error.
    
    Not used in Morphew et al. 2026
    """
    def __init__(self, weight_factor=100.0):
        """
        Args:
            weight_factor (float): The factor to multiply the loss by for non-zero elements in the label tensor.
        """
        super(WeightedMSELoss, self).__init__()
        self.weight_factor = weight_factor

    def forward(self, outputs, labels):
        weights = torch.ones_like(labels)
        weights[labels > 1e-6] = self.weight_factor # Use a small epsilon
        loss = torch.mean(weights * (outputs - labels) ** 2)
        return loss

class RegressionFocalLoss(nn.Module):
    """
    An adaptation of Focal Loss for regression tasks.
    This version is generic. It focuses training on "hard" examples by weighting the loss
    by the magnitude of the error itself.
    """
    def __init__(self, gamma=1.5):
        """
        Args:
            gamma (float): The focusing parameter. Higher values give more weight to hard-to-predict examples.
        """
        super(RegressionFocalLoss, self).__init__()
        self.gamma = gamma

    def forward(self, outputs, labels):
        abs_error = torch.abs(outputs - labels)
        # The focal modulation term. Errors that are already small will have a very small weight,
        # while large errors will have a weight closer to 1.
        focal_weight = abs_error.pow(self.gamma)
        loss = torch.mean(focal_weight * F.l1_loss(outputs, labels, reduction='none'))
        return loss


# helper function for SSIM
def gaussian(window_size, sigma):
    gauss = torch.Tensor([exp(-(x - window_size//2)**2/float(2*sigma**2)) for x in range(window_size)])
    return gauss/gauss.sum()

# helper function for SSIM
def create_window(window_size, channel):
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
    return window

class SSIM(nn.Module):
    """
    SSIM calculation
    """
    def __init__(self, window_size=11, size_average=True):
        super(SSIM, self).__init__()
        self.window_size = window_size
        self.size_average = size_average
        self.channel = 1
        self.window = create_window(window_size, self.channel)

    def forward(self, img1, img2):
        (_, channel, _, _) = img1.size()
        if channel == self.channel and self.window.data.type() == img1.data.type():
            window = self.window
        else:
            window = create_window(self.window_size, channel)
            if img1.is_cuda: window = window.cuda(img1.get_device())
            window = window.type_as(img1)
            self.window = window
            self.channel = channel

        mu1 = F.conv2d(img1, window, padding=self.window_size//2, groups=channel)
        mu2 = F.conv2d(img2, window, padding=self.window_size//2, groups=channel)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1*mu2
        sigma1_sq = F.conv2d(img1*img1, window, padding=self.window_size//2, groups=channel) - mu1_sq
        sigma2_sq = F.conv2d(img2*img2, window, padding=self.window_size//2, groups=channel) - mu2_sq
        sigma12 = F.conv2d(img1*img2, window, padding=self.window_size//2, groups=channel) - mu1_mu2
        C1, C2 = 0.01**2, 0.03**2
        ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2))/((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))
        return ssim_map.mean() if self.size_average else ssim_map.mean(1).mean(1).mean(1)

class SSIMLoss(nn.Module):
    """ Wraps the SSIM calculation in a loss function: Loss = 1 - SSIM. """
    def __init__(self, window_size=11, size_average=True):
        super(SSIMLoss, self).__init__()
        self.ssim = SSIM(window_size, size_average)

    def forward(self, outputs, labels):

        # This handles variations between training and validation batches.
        if outputs.dim() == 6:
            # Squeeze out an extra dimension, often added by dataloaders for batch size 1
            outputs = outputs.squeeze(0)
        if labels.dim() == 6:
            labels = labels.squeeze(0)

        # This makes the function robust to being called on a slice or a full tensor.
        if outputs.dim() == 4:
            outputs = outputs.unsqueeze(-1)
        if labels.dim() == 4:
            labels = labels.unsqueeze(-1)

        # Reshape for SSIM calculation: (batch, X, Y, time, channel) -> (batch*time, channel, Y, X)
        outputs = outputs.permute(0, 3, 4, 2, 1)
        labels = labels.permute(0, 3, 4, 2, 1)
        b, t, c, h, w = outputs.shape
        outputs_reshaped = outputs.reshape(b * t, c, h, w)
        labels_reshaped = labels.reshape(b * t, c, h, w)
        ssim_val = self.ssim(outputs_reshaped, labels_reshaped)
        return 1.0 - ssim_val # loss will be the compliment to SSIM, not SSIM itself

class SafeSSIMLoss(nn.Module):
    """
    A wrapper for SSIMLoss that safely handles z-scored data by re-normalizing
    each batch to the [0, 1] range internally before SSIM calculation.
    """
    def __init__(self, window_size=11):
        super(SafeSSIMLoss, self).__init__()
        self.ssim_loss = SSIMLoss(window_size=window_size) # original ssim class
        self.epsilon = 1e-8

    def forward(self, outputs, labels):
        # The data shape is (batch, z, x, time, channel)
        # This handles variations between training and validation batches.
        if outputs.dim() == 6:
            # Squeeze out the extra dimension, often added by dataloaders for batch size 1
            outputs = outputs.squeeze(0)
        if labels.dim() == 6:
            labels = labels.squeeze(0)

        # This makes the function robust to being called on a slice or the full tensor.
        if outputs.dim() == 4:
            outputs = outputs.unsqueeze(-1)
        if labels.dim() == 4:
            labels = labels.unsqueeze(-1)
        outputs_renorm = torch.zeros_like(outputs)
        labels_renorm = torch.zeros_like(labels)
        
        # Re-normalize each channel and each sample in the batch independently
        for i in range(outputs.shape[0]): # Iterate over samples in batch
            for j in range(outputs.shape[4]): # Iterate over channels
                out_slice = outputs[i, ..., j]
                lbl_slice = labels[i, ..., j]
                
                # Get the dynamic range of the combined slices
                min_val = min(out_slice.min(), lbl_slice.min())
                max_val = max(out_slice.max(), lbl_slice.max())
                range_val = max_val - min_val
                
                if range_val > self.epsilon:
                    outputs_renorm[i, ..., j] = (out_slice - min_val) / range_val
                    labels_renorm[i, ..., j] = (lbl_slice - min_val) / range_val
        
        # Now, the re-normalized tensors are safe to pass to SSIM
        return self.ssim_loss(outputs_renorm, labels_renorm)


class GradientLoss(nn.Module):
    """
    Computes the L1 loss between the gradients of the prediction and the label.

    Not used in Morphew et al. 2026
    """
    def __init__(self):
        super(GradientLoss, self).__init__()
        self.loss = nn.L1Loss()

    def forward(self, outputs, labels):
        # We expect a 5D tensor: (batch, z, x, time, channel)
        # We will calculate gradients along the z (dim=1) and x (dim=2) axes.
        # This handles variations between training and validation batches.
        if outputs.dim() == 6:
            # Squeeze out the extra dimension, often added by dataloaders for batch size 1
            outputs = outputs.squeeze(0)
        if labels.dim() == 6:
            labels = labels.squeeze(0)

        # This makes the function robust to being called on a slice or the full tensor.
        if outputs.dim() == 4:
            outputs = outputs.unsqueeze(-1)
        if labels.dim() == 4:
            labels = labels.unsqueeze(-1)
        # Gradient in z-direction
        grad_z_outputs = outputs[:, 1:, :, :, :] - outputs[:, :-1, :, :, :]
        grad_z_labels = labels[:, 1:, :, :, :] - labels[:, :-1, :, :, :]

        # Gradient in x-direction
        grad_x_outputs = outputs[:, :, 1:, :, :] - outputs[:, :, :-1, :, :]
        grad_x_labels = labels[:, :, 1:, :, :] - labels[:, :, :-1, :, :]

        # Calculate L1 loss for each gradient direction and add them
        loss_z = self.loss(grad_z_outputs, grad_z_labels)
        loss_x = self.loss(grad_x_outputs, grad_x_labels)

        return loss_x + loss_z


class FrequencyLoss(nn.Module):
    """
    Computes the L1 loss between the magnitudes of the Fourier spectra
    of the prediction and the label.

    Not used in Morphew et al. 2026
    """
    def __init__(self):
        super(FrequencyLoss, self).__init__()
        self.loss = nn.L1Loss()

    def forward(self, outputs, labels):
        # We expect a 5D tensor: (batch, z, x, time, channel)
        # Apply FFT along the spatial dimensions (z and x)
        outputs_fft = torch.fft.rfftn(outputs, dim=(1, 2))
        labels_fft = torch.fft.rfftn(labels, dim=(1, 2))
        
        # Calculate the loss on the magnitude of the spectra
        loss = self.loss(torch.abs(outputs_fft), torch.abs(labels_fft))
        return loss

# ==========================================================================================
# SECTION 2: THE LOSS COMBINER
# This master class takes a list of loss definitions and combines them.
# ==========================================================================================
class MultiChannelLoss(nn.Module):
    """
    A master loss function to combine different loss functions on different channels.
    """
    def __init__(self, loss_definitions):
        """
        Args:
            loss_definitions (list): A list of tuples. Each tuple should contain:
                (loss_function (nn.Module), channel_index (int or 'all'), weight (float))
        """
        super(MultiChannelLoss, self).__init__()
        self.loss_definitions = loss_definitions

    def forward(self, outputs, labels):
        total_loss = 0.0
        for loss_fn, channel, weight in self.loss_definitions:
            if channel == 'all':
                # Apply loss to the entire tensor
                pred_target = outputs
                label_target = labels
            else:
                # Apply loss to a specific channel
                pred_target = outputs[..., channel]
                label_target = labels[..., channel]
            
            # Calculate the weighted loss and add to the total
            loss = loss_fn(pred_target, label_target)
            total_loss += weight * loss
        return total_loss


# Some sample options for how to construct a combined loss function
# "Option Paper" was the function construction used in Morphew et al. 2026

#Option 1: We apply standard MSE to the head channel (0) and WeightedMSE to the concentration channel (1).

#definitions = [
#    (nn.MSELoss(), 0, 1.0),  # (loss_fn, channel_index, weight)
#    (WeightedMSELoss(weight_factor=150.0), 1, 1.0)
#]
# loss_fn = MultiChannelLoss(definitions)


#Option 2: A complex combination of L1, SSIM, and Focal Loss 
# - L1 loss on the head channel (0)
# - SSIM loss on the whole tensor to preserve structure
# - Regression Focal Loss on the concentration channel (1) to focus on the plume
#definitions = [
#    (nn.MSELoss(), 'all', 1), # L1 loss on head channel with weight 0.5
#    (SafeSSIMLoss(), 1, 1), # SSIM on all channels with weight 1.0
#    (RegressionFocalLoss(gamma=1.5), 1, 1) # Focal Loss on conc channel with weight 1.5
#]

#Option 3: Just SSIM 
# definitions_3 = [
#     (SSIMLoss(), 'all', 1.0)
# ]
# loss_fn = MultiChannelLoss(definitions)

#Option Paper: MSE + SSIM + Focal
definitions = [
    (nn.MSELoss(), 'all', 1), # L1 loss on head channel with weight 0.5
    (SafeSSIMLoss(), 1, 1), # SSIM on all channels with weight 1.0
    (RegressionFocalLoss(gamma=1.5), 1, 1) # Focal Loss on conc channel with weight 1.5
]
loss_fn = MultiChannelLoss(definitions)

Optimizer
=========

The optimizer used in Morphew et al. 2026 is Adam with an initial learning rate of 0.001, but this can be changed as desired.


In [ ]:
lr = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

Training
==================

The code will now train in a step wise fashion. If training or validations exceed a certain threshold, they will be saved out as PNGs so that the model can be analyzed for weaknesses.


In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
def plot_channel_grid(data, sample_idx, channel_idx, title, prefix, folder_name, vmin=None, vmax=None):
    """
    Plots a grid of images for a specific sample and channel.
    'data' should be a numpy array of shape (samples, z, x, time, channels)
    """
    sample_data = data[sample_idx, ..., channel_idx]

    num_timesteps = sample_data.shape[2] 
    rows, cols = 3, 4
    
    fig, axes = plt.subplots(rows, cols, figsize=(13.33, 7.5), constrained_layout=True)
    axes_flat = axes.flatten()

    if vmin is None: vmin = np.nanmin(sample_data)
    if vmax is None: vmax = np.nanmax(sample_data)

    im = None 
    for t in range(min(num_timesteps, 11)): # Cap at 11 to leave room for cbar
        image_slice = sample_data[:, :, t]
        ax = axes_flat[t]
        im = ax.imshow(image_slice, cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set_title(f"{prefix} T{t+1}")
        ax.axis('off') 

    # Hide unused axes
    for i in range(num_timesteps, len(axes_flat)):
        axes_flat[i].axis('off')

    # Colorbar in the last slot
    parent_cbar_ax = axes_flat[-1]
    cbar_ax = inset_axes(parent_cbar_ax, width="90%", height="10%", loc='center')
    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal') 
    
    label = 'head (ft)' if channel_idx == 0 else 'concentration (ppb)'
    cbar.set_label(label)
        
    # Save Logic
    os.makedirs(folder_name, exist_ok=True)
    channel_label = 'Head' if channel_idx == 0 else 'Conc'
    save_path = f'{folder_name}/step_{title}_{channel_label}_{prefix}_idx{sample_idx}.png'
    
    plt.savefig(save_path, dpi=200)
    plt.close(fig)

    return vmin, vmax

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter('runs/neptune_trainer_{}'.format(timestamp))
#Training and Validation Loop (Step-based)
print("\nStarting step-based training...")
training_iter = iter(training_loader)
validation_iter = iter(validation_loader)
best_vloss = 1e20
for step in range(MAX_STEPS):
    model.train()
    optimizer.zero_grad()
    
    # Get a batch of data. Use a try-except block to handle the end of an epoch
    try:
        inputs_data, labels_data = next(training_iter)
    except StopIteration:
        # If the iterator is exhausted, re-initialize it for the next epoch
        training_iter = iter(training_loader)
        inputs_data, labels_data = next(training_iter)
    
    inputs = inputs_data.squeeze().to(device)
    labels = labels_data.squeeze().to(device)
    # Make predictions, compute loss and step the optimizer
    outputs = model(inputs.float())
    loss = loss_fn(outputs, labels)
    loss.backward()
    optimizer.step()
    
    # Log training loss per step
    writer.add_scalar('Training Loss/step', loss.item(), step)
    
    # Validation logic
    if (step + 1) % VALIDATION_FREQUENCY == 0:
        print(f"Step {step + 1}/{MAX_STEPS} - Running validation...")
        model.eval()
        running_vloss = 0.0
        
        with torch.no_grad():
            try:
                v_inputs_data, v_labels_data = next(validation_iter)
            except StopIteration:
                validation_iter = iter(validation_loader)
                v_inputs_data, v_labels_data = next(validation_iter)
                
            v_inputs = v_inputs_data.squeeze().to(device)
            v_labels = v_labels_data.squeeze().to(device)
            
            v_outputs = model(v_inputs)
            vloss = loss_fn(v_outputs, v_labels)
            running_vloss += vloss.item()
        
        avg_vloss = running_vloss

        # Detection logic for bad samples
        
        # if we score poorly on either the training or validation sample when running validation, we'll save out the offender as a png
        if avg_vloss > LOSS_THRESHOLD:
            print(f"⚠️ Validation Loss Spike Detected at step {step+1}: {avg_vloss:.4f}")
            
            # Convert to numpy for plotting
            # Shape assumed: (Batch, X, Y, Time, Channels)
            preds_np = v_outputs.detach().cpu().numpy()
            gt_np = v_labels.detach().cpu().numpy()
            
            for ch in [0, 1]:
                for idx in range(BATCH_SIZE):
                    # Plot Ground Truth
                    vmin, vmax = plot_channel_grid(gt_np, idx, ch, 
                                        title=str(step+1), prefix="GT_VAL",
                                        folder_name=PLOT_DIR)
                    # Plot Prediction
                    plot_channel_grid(preds_np, idx, ch, 
                                        title=str(step+1), prefix="PRED_VAL", 
                                        folder_name=PLOT_DIR, vmin=vmin, vmax=vmax)
        if loss.item() > LOSS_THRESHOLD:
            print(f"⚠️ Training Loss Spike Detected at step {step+1}: {loss.item():.4f}")
            
            preds_np = outputs.detach().cpu().numpy()
            gt_np = labels.detach().cpu().numpy()
        
            for ch in [0, 1]:
                for idx in range(BATCH_SIZE):
                    # Plot Ground Truth
                    vmin, vmax = plot_channel_grid(gt_np, idx, ch, 
                                        title=str(step+1), prefix="GT_TRAIN", 
                                        folder_name=PLOT_DIR)
                    # Plot Prediction
                    plot_channel_grid(preds_np, idx, ch, 
                                        title=str(step+1), prefix="PRED_TRAIN", 
                                        folder_name=PLOT_DIR, vmin=vmin, vmax=vmax)
        writer.add_scalars(
            'Training vs. Validation Loss',
            {   'Training': loss.item(),
                'Validation': avg_vloss},
            step
        )
        print(f"Training loss at step {step + 1}: {loss.item():.4f}")
        print(f"Validation Loss at step {step + 1}: {avg_vloss:.4f}")
        
    
    # Save a checkpoint every N step
    if (step + 1) % SAVE_FREQUENCY == 0:
        model_path_snapshot = os.path.join(CHECKPOINT_PATH, f'model_snapshot_{step+1}.pt')
        torch.save(model.state_dict(), model_path_snapshot)
        print(f"Saved model snapshot at step {step+1}.")
        if avg_vloss < best_vloss:
            print("validation loss better than best. Saving as best snapshot.")
            best_vloss = avg_vloss
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_PATH, f'model_best_snapshot.pt'))

writer.close()
print("\nTraining complete.")

# To visualize results on the test dataset, look at the "testing-performance-plots.ipynb" notebook.

# This requires that both a UFNO and AUNET model are trained.